<!--
    Copyright (C) 2026 The AiQuokkaWannabe Authors <https://github.com/ai-quokka-wannabe/iso-lab>

    This program is free software: you can redistribute it and/or modify it under the terms of
    the GNU General Public License as published by the Free Software Foundation, either version
    3 of the License, or (at your option) any later version.

    This program is distributed in the hope that it will be useful, but WITHOUT ANY WARRANTY;
    without even the implied warranty of MERCHANTABILITY or FITNESS FOR A PARTICULAR PURPOSE.
    See the GNU General Public License for more details.

    You should have received a copy of the GNU General Public License along with this program.
    If not, see <https://www.gnu.org/licenses/>.
-->

# The Lab's Diagrams

Every diagram in `docs/figures/` whose name starts `diagram_` is drawn here, as plain SVG written by Python with the
standard library alone, so that it needs no tool beyond the notebook's own pins (`notebooks/requirements.txt`) and
diffs as text. The drawing is deterministic: integer coordinates, a fixed element order, no timestamps or random
identifiers, so running this notebook again on an unchanged tree changes no byte of any figure. The last cell checks
that, by drawing every diagram twice and comparing the bytes.

Run with `jupyter nbconvert --to notebook --execute --inplace notebooks/figures.ipynb`.


In [1]:
import os
from html import escape

# nbconvert runs a notebook from its own directory; the figures live under the repository's root.
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
FIGURES = "docs/figures"
FONT = "DejaVu Sans, Verdana, Arial, sans-serif"

# One palette for every diagram: a repository or a role keeps its colour from figure to figure.
PALETTE = {
    "world": ("#e8f1fb", "#2f6db3"),   # Master Control, the world server
    "wire": ("#eef7ee", "#3d8b40"),    # Link, the wire
    "grid": ("#fdf1e3", "#c46f12"),    # tron-grid-lite, the Grid
    "brain": ("#f6ecfa", "#8a3fa8"),   # a Program, a creature's brain
    "lab": ("#fff8d9", "#9a7b00"),     # iso-lab's own tools and documents
    "data": ("#eef0f2", "#4d5a66"),    # data from outside: the real worms
    "note": ("#ffffff", "#7a7a7a"),    # a plain note
}


def species(line):
    """Escape a line of text, setting the worm's binomial name in italics, as a species name is written."""
    name = "C. elegans"
    return f'<tspan font-style="italic">{name}</tspan>'.join(escape(part) for part in line.split(name))


class Svg:
    """A tiny SVG writer: rounded boxes, arrows and text, every number an integer."""

    def __init__(self, width, height, title):
        self.width, self.height, self.title = width, height, title
        self.parts = []

    def box(self, x, y, w, h, title, lines=(), role="note", dashed=False):
        fill, stroke = PALETTE[role]
        dash = ' stroke-dasharray="6 4"' if dashed else ""
        self.parts.append(
            f'<rect x="{x}" y="{y}" width="{w}" height="{h}" rx="10" fill="{fill}" stroke="{stroke}" stroke-width="2"{dash}/>'
        )
        cx = x + w // 2
        ty = y + 26 if lines else y + h // 2 + 6
        self.parts.append(
            f'<text x="{cx}" y="{ty}" text-anchor="middle" font-size="16" font-weight="bold" fill="#1d1d1d">{escape(title)}</text>'
        )
        for i, line in enumerate(lines):
            self.parts.append(
                f'<text x="{cx}" y="{ty + 22 + 19 * i}" text-anchor="middle" font-size="13" fill="#333333">{species(line)}</text>'
            )

    def arrow(self, points, label=None, label_at=None, dashed=False, both=False, colour="#444444"):
        d = "M " + " L ".join(f"{x} {y}" for x, y in points)
        dash = ' stroke-dasharray="6 4"' if dashed else ""
        start = ' marker-start="url(#head)"' if both else ""
        self.parts.append(
            f'<path d="{d}" fill="none" stroke="{colour}" stroke-width="2"{dash}{start} marker-end="url(#head)"/>'
        )
        if label:
            lx, ly = label_at
            self.parts.append(
                f'<text x="{lx}" y="{ly}" text-anchor="middle" font-size="13" font-style="italic" fill="#333333">{escape(label)}</text>'
            )

    def text(self, x, y, s, size=13, anchor="middle", bold=False, colour="#333333"):
        weight = ' font-weight="bold"' if bold else ""
        self.parts.append(
            f'<text x="{x}" y="{y}" text-anchor="{anchor}" font-size="{size}"{weight} fill="{colour}">{escape(s)}</text>'
        )

    def render(self):
        head = (
            f'<svg xmlns="http://www.w3.org/2000/svg" width="{self.width}" height="{self.height}" '
            f'viewBox="0 0 {self.width} {self.height}" font-family="{FONT}">\n'
            f"<title>{escape(self.title)}</title>\n"
            '<defs><marker id="head" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="8" markerHeight="8" '
            'orient="auto-start-reverse"><path d="M 0 0 L 10 5 L 0 10 z" fill="#444444"/></marker></defs>\n'
            f'<rect x="0" y="0" width="{self.width}" height="{self.height}" fill="#ffffff"/>\n'
        )
        return head + "\n".join(self.parts) + "\n</svg>\n"


def save(name, svg):
    path = os.path.join(FIGURES, name)
    with open(path, "w", encoding="utf-8", newline="\n") as f:
        f.write(svg.render())
    return path


DIAGRAMS = {}


def diagram(name):
    def register(draw):
        DIAGRAMS[name] = draw
        return draw
    return register


## How the Pieces Fit

The organisation's repositories and the lab's own tools, and what flows between them. A creature's brain - a Program -
sees the world only through the senses the Grid renders for its body, and acts only through that body; the world
itself is Master Control's, reached over Link. What a life leaves behind - the Disk and the log - and the recordings
of real worms meet in the lab's tools, where the benchmark compares them. A dashed box or arrow is planned, not built.


In [2]:
@diagram("diagram_pieces.svg")
def pieces():
    s = Svg(1100, 620, "How the pieces fit")
    s.text(550, 34, "How the pieces fit", size=22, bold=True, colour="#1d1d1d")
    s.text(550, 58, "a creature lives on the Grid; the lab compares its life with real worms")

    # The world, left to right: Master Control - Link - the Grid - a Program.
    s.box(30, 90, 220, 120, "Master Control", ["the world server: physics,", "food, the tick, the logs"], role="world")
    s.box(320, 110, 150, 80, "Link", ["the wire"], role="wire")
    s.box(540, 90, 230, 120, "The Grid", ["renders each body's senses", "and applies its actions"], role="grid")
    s.box(860, 90, 210, 120, "A Program", ["a creature's brain,", "e.g. the placeholder worm"], role="brain")
    s.arrow([(250, 150), (318, 150)], both=True)
    s.arrow([(470, 150), (538, 150)], both=True)
    s.arrow([(770, 125), (858, 125)], "senses", (814, 115))
    s.arrow([(858, 175), (772, 175)], "actions", (814, 198))

    # What a life leaves behind.
    s.box(30, 300, 220, 100, "Disk and log", ["a life, recorded"], role="world")
    s.arrow([(140, 210), (140, 298)], "writes", (172, 260))
    s.box(30, 480, 220, 100, "Clu", ["replays a log and checks", "it gives the same world"], role="world")
    s.arrow([(140, 400), (140, 478)], "replay", (170, 444))

    # The real worms and the lab's tools.
    s.box(860, 300, 210, 100, "Real worms", ["recordings of C. elegans", "(Zenodo, public)"], role="data")
    s.box(540, 300, 230, 100, "fetch, features", ["the lab's Rust tools:", "download, then measure"], role="lab")
    s.arrow([(858, 350), (772, 350)], "download", (814, 340))
    s.arrow([(250, 350), (538, 350)], "read the Disk", (394, 340), dashed=True)
    s.text(394, 372, "(Disk reader planned)", size=12, colour="#7a7a7a")
    s.box(540, 480, 230, 100, "The benchmark", ["is the Grid worm's behaviour", "within the real worms'?"], role="lab", dashed=True)
    s.arrow([(655, 400), (655, 478)], "both sets of measures", (740, 444), dashed=True)
    s.text(655, 600, "(planned: not yet built or frozen)", size=12, colour="#7a7a7a")
    return s


## A Night's Cycle

How each task of a night lands, from `.claude/CLAUDE.md` § *Each run, in order*. A task is landed or honestly parked:
after three rounds of fixes that still end in *changes needed*, the pull request stays open and the task goes under
STATE.md's *Blocked*, with the next idea to try.


In [3]:
@diagram("diagram_night.svg")
def night():
    s = Svg(1100, 700, "A night's cycle")
    s.text(550, 34, "A night's cycle", size=22, bold=True, colour="#1d1d1d")
    s.text(550, 58, "each task lands through the same steps, or is honestly parked")
    W, H = 230, 70
    # The main line, left to right then down: read, pick, branch, work, check, write down.
    s.box(30, 90, W, H, "1. Start from main", ["fetch, check out, pull"], role="lab")
    s.box(300, 90, W, H, "2. Read", ["STATE.md, the plan, the journal"], role="lab")
    s.box(570, 90, W, H, "3. Pick one task", ["about 45 minutes; split if bigger"], role="lab")
    s.box(840, 90, W, H, "4. Branch", ["claude/<short-name>"], role="lab")
    s.arrow([(260, 125), (298, 125)])
    s.arrow([(530, 125), (568, 125)])
    s.arrow([(800, 125), (838, 125)])
    s.arrow([(955, 160), (955, 208)])
    s.box(840, 210, W, H, "5. Do the work", ["pre-register; record every run"], role="brain")
    s.box(570, 210, W, H, "6-7. Check", ["review as a sceptic; lint, tests"], role="brain")
    s.box(300, 210, W, H, "8. Write it down", ["STATE.md and the journal"], role="brain")
    s.box(30, 210, W, H, "9. Pull request", ["commit, push, open it"], role="brain")
    s.arrow([(838, 245), (802, 245)])
    s.arrow([(568, 245), (532, 245)])
    s.arrow([(298, 245), (262, 245)])
    s.arrow([(145, 280), (145, 338)])
    # The gate: CI and a second agent.
    s.box(30, 340, W, H, "10. CI", ["the CI Success gate"], role="grid")
    s.box(300, 340, W, H, "11. Second agent", ["a fresh verifier reviews"], role="grid")
    s.arrow([(260, 375), (298, 375)])
    s.arrow([(415, 410), (415, 478)])
    s.text(405, 450, "verified, CI green", size=13, anchor="end", colour="#333333")
    s.box(300, 480, W, H, "12. Merge", ["the branch is deleted"], role="wire")
    s.box(640, 340, 250, H, "Fix what it found", ["or answer why not"], role="note")
    s.arrow([(530, 365), (638, 365)], "changes needed", (585, 355))
    s.arrow([(700, 410), (700, 440), (470, 440), (470, 412)])
    s.text(585, 432, "push, and it looks again", size=13, colour="#333333")
    s.box(640, 480, 250, 88, "13. Park it", ["after three rounds: stays open,", "recorded under Blocked"], role="note", dashed=True)
    s.arrow([(830, 410), (830, 478)], dashed=True)
    s.text(842, 450, "after 3 rounds", size=12, anchor="start", colour="#7a7a7a")
    s.box(300, 610, W, H, "14. Next task, or stop", ["back to 1 until the hour"], role="lab")
    s.arrow([(415, 550), (415, 608)])
    s.arrow([(765, 568), (765, 645), (532, 645)], dashed=True)
    s.arrow([(298, 645), (14, 645), (14, 125), (28, 125)])
    s.text(156, 635, "the next task", size=13, colour="#333333")
    return s


## The Benchmark's Pipeline

How a verdict on the Grid's worm will be reached, from [docs/PLAN.md](../docs/PLAN.md) Phase 0 and
[docs/benchmark/FEATURES.md](../docs/benchmark/FEATURES.md). The real worms' recordings and the Grid worm's lives
pass through the same measures, and the benchmark, frozen before any brain is scored, compares them. A dashed box is
planned, not yet built.


In [4]:
@diagram("diagram_benchmark.svg")
def benchmark():
    s = Svg(1190, 520, "The benchmark's pipeline")
    s.text(595, 34, "The benchmark's pipeline", size=22, bold=True, colour="#1d1d1d")
    s.text(595, 58, "real worms and the Grid's worm, measured the same way, compared by rules fixed in advance")
    W, H = 250, 90
    # Real worms, top row.
    s.box(30, 90, W, H, "Real worms", ["recordings of wild-type", "C. elegans (public)"], role="data")
    s.box(320, 90, W, H, "fetch-block", ["download a block;", "keep files whose MD5 matches"], role="lab")
    s.box(610, 90, W, H, "features", ["skeletons to speed, states,", "durations and rates"], role="lab")
    s.box(900, 90, W, H, "Reference statistics", ["the real worms' spread", "for every measure"], role="lab", dashed=True)
    s.arrow([(280, 135), (318, 135)])
    s.arrow([(570, 135), (608, 135)])
    s.arrow([(860, 135), (898, 135)])
    # The Grid's worm, bottom row.
    s.box(30, 300, W, H, "A Program's lives", ["on the Grid, many seeds"], role="brain")
    s.box(320, 300, W, H, "Disk", ["each life, recorded", "by Master Control"], role="world")
    s.box(610, 300, W, H, "features", ["the same measures,", "via a Disk reader"], role="lab", dashed=True)
    s.box(900, 300, W, H, "Grid worm statistics", ["the same measures", "over its lives"], role="lab", dashed=True)
    s.arrow([(280, 345), (318, 345)])
    s.arrow([(570, 345), (608, 345)], dashed=True)
    s.arrow([(860, 345), (898, 345)], dashed=True)
    # The comparison.
    s.box(900, 420, W, 80, "Verdict", ["per measure, by frozen rules"], role="grid", dashed=True)
    s.arrow([(1150, 135), (1168, 135), (1168, 460), (1152, 460)], dashed=True)
    s.arrow([(1025, 390), (1025, 418)], dashed=True)
    s.text(735, 455, "the benchmark is frozen", size=13, colour="#333333")
    s.text(735, 474, "before any brain is scored", size=13, colour="#333333")
    return s


## Drawing, and the Check That Nothing Moves

Every diagram is drawn twice and the two drawings compared, which catches a drawing that varies within one run. Before
the new drawing is written, it is compared with the file already in `docs/figures/`, and the cell reports each figure
as *unchanged*, *changed* or *new*. A figure reported *changed* when its code did not change means the drawing depends
on something besides this notebook's code, which only a separate run can show: the check that matters is the
notebook run again from a fresh kernel, then `git diff --exit-code docs/figures`, which must report nothing.


In [5]:
import hashlib

for name, draw in sorted(DIAGRAMS.items()):
    first, second = draw().render(), draw().render()
    assert first == second, f"{name} differs between two drawings in one run"
    path = os.path.join(FIGURES, name)
    before = open(path, encoding="utf-8").read() if os.path.exists(path) else None
    status = "new" if before is None else ("unchanged" if before == first else "changed")
    with open(path, "w", encoding="utf-8", newline="\n") as f:
        f.write(first)
    digest = hashlib.sha256(first.encode("utf-8")).hexdigest()
    print(f"{path}: {status}, {len(first.encode('utf-8'))} bytes, sha256 {digest}")


docs/figures/diagram_benchmark.svg: unchanged, 5602 bytes, sha256 57a27446d33585e3f1d041d7011769891ff85094d96ad206e7a2d15bf57dce1e
docs/figures/diagram_night.svg: unchanged, 7588 bytes, sha256 b046f9ce005efe107eeb2ec39fbe469e54383a4045bb1d56b2942ab1b6e03c7f
docs/figures/diagram_pieces.svg: unchanged, 6371 bytes, sha256 656b21dac3684de3c282be2ec77232f5cf7bad19ac6f06c31e56b83e56660d4c
